# One-shot & few-shot learning

_Few-shot & Transfer Learning_

**Learn a brand-new class from a single picture, or just a handful, by averaging them into a prototype.**

Usually a model needs hundreds or thousands of labeled examples per class to learn it.

       Few-shot learning breaks that rule. You show the model a brand-new class with only a handful of labeled examples and it recognizes more of that class right away.

---

This notebook is a practice scaffold for the **One-shot & few-shot learning** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Visualize the data & results

_On real handwritten digits, how much does few-shot accuracy improve as you go from one-shot to many shots per class?_

### Step 1 — Load the digits and learn an embedding

We take the real handwritten digits, scale the pixels to the 0–1 range, and restrict to a 5-way task (classes 0–4). Then we learn an embedding *once* with Neighborhood Components Analysis, a supervised metric-learning method that pulls same-class digits together and pushes different classes apart. Every example is mapped through this fixed embedding `f(x)` before we ever form prototypes — the quality of few-shot learning rides entirely on this representation.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NeighborhoodComponentsAnalysis

digits = load_digits()                       # 1797 real 8x8 handwritten digits
X = digits.data / 16.0
y = digits.target

classes = [0, 1, 2, 3, 4]                     # a 5-way task
mask = np.isin(y, classes)
Xc, yc = X[mask], y[mask]

# Embed ONCE with supervised metric learning (groups similar digits together).
Xs = StandardScaler().fit_transform(Xc)
nca = NeighborhoodComponentsAnalysis(n_components=10, random_state=0, max_iter=50)
E = nca.fit_transform(Xs, yc)                 # the embedding f(x)

### Step 2 — Run few-shot episodes for each number of shots

For each shot count `K`, we run 200 random episodes. In every episode we sample `K` support examples per class, average their embeddings into one **prototype** per class, then classify the remaining held-out queries by nearest prototype in embedding space. Averaging over many random draws gives a stable estimate of how accuracy depends on `K`. This is the core Prototypical-Network rule: predict the class whose mean embedding is closest to the query.

In [ ]:
shots = [1, 2, 5, 10, 20]
rng = np.random.default_rng(0)
acc_by_K = []

for K in shots:
    accs = []
    for _ in range(200):                      # average over 200 random episodes
        # pick K support examples per class
        chosen = {c: set(rng.choice(np.where(yc == c)[0], K, replace=False))
                  for c in classes}
        # prototype c_k = mean embedding of that class's support set
        protos = np.stack([E[list(chosen[c])].mean(axis=0)
                           for c in classes])
        correct = total = 0
        for c in classes:                     # queries = the rest of the class
            pool = [i for i in np.where(yc == c)[0] if i not in chosen[c]]
            for qi in rng.choice(pool, min(20, len(pool)), replace=False):
                d = ((E[qi] - protos) ** 2).sum(axis=1)        # nearest prototype
                correct += (classes[d.argmin()] == c)
                total += 1
        accs.append(correct / total)
    acc_by_K.append(np.mean(accs))

### Step 3 — Read off how accuracy grows with shots

Finally we print accuracy against the number of shots per class. Notice the shape of the curve: one-shot already does far better than chance, and each extra shot helps but with shrinking returns — going from 1 to 2 shots gains a lot, while 10 to 20 barely moves. More support examples give a steadier prototype, but the embedding ultimately caps how high accuracy can climb.

In [ ]:
print(list(zip(shots, np.round(acc_by_K, 3))))
# -> [(1, 0.768), (2, 0.847), (5, 0.902), (10, 0.925), (20, 0.934)]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In a 3-way 5-shot episode, how many labeled examples are in the support set, and what is $K$ for a one-shot version?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read off N and K. "3-way" means $N = 3$ classes. "5-shot" means $K = 5$ examples per class. — _N-way K-shot fully describes the task size._
- Support size = $N \times K = 3 \times 5 = 15$ labeled examples. — _Each of the 3 classes contributes its own 5 support examples._
- One-shot means $K = 1$. — _One-shot is just the special case of K-shot with a single example per class._

**Answer:** 15 support examples; one-shot is $K = 1$.

</details>

**Problem 2.** Class P has support embeddings $[2, 0]$ and $[4, 0]$. Class Q has support embeddings $[0, 4]$ and $[0, 6]$. A query embeds to $f(x) = [1, 1]$. Which class does a Prototypical Network predict?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Prototype of P: average the two vectors. $c_P = \frac{[2,0] + [4,0]}{2} = [3, 0]$. — _The prototype is the mean embedding of that class's support set._
- Prototype of Q: $c_Q = \frac{[0,4] + [0,6]}{2} = [0, 5]$. — _Same averaging rule, applied to class Q._
- Squared distance to P: $\|[1,1] - [3,0]\|^2 = (-2)^2 + 1^2 = 4 + 1 = 5$. — _Squared distance sums the squared gaps in each coordinate._
- Squared distance to Q: $\|[1,1] - [0,5]\|^2 = 1^2 + (-4)^2 = 1 + 16 = 17$. — _Compare the query to every prototype._
- Pick the smallest: $5 < 17$, so prototype P is nearest. — _$\hat{y} = \arg\min_k \| f(x) - c_k \|^2$ chooses the nearest prototype._

**Answer:** Class P (distance 5 to P beats 17 to Q).

</details>

**Problem 3.** Why does a Prototypical Network barely work if the embedding $f$ has not already been trained to group similar inputs together?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what the prototype depends on. Each prototype $c_k$ is a mean of embeddings $f(x_i)$. — _If those embeddings are meaningless, their average is meaningless too._
- Recall the decision rule: nearest prototype by distance in embedding space. — _The whole method assumes distance in that space reflects class similarity._
- If $f$ does not separate classes, the class blobs overlap and prototypes land in tangled spots. — _Nearest-prototype then guesses near chance because the geometry carries no class information._

**Answer:** The method does no per-class training; it relies entirely on $f$ placing same-class inputs near each other. A bad embedding gives overlapping blobs and near-random predictions.

</details>